# 02. Prompting Agéntico

**Nivel:** 🟡 Intermedio  
**Tiempo estimado:** 90-120 minutos  
**Prerequisitos:** [01. Introducción a LLM Agents](01-intro-llm-agents.ipynb)

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook, podrás:
- Implementar Chain-of-Thought (CoT) prompting para razonamiento paso a paso
- Usar Tree-of-Thought (ToT) para explorar múltiples caminos de razonamiento
- Aplicar Self-Consistency para mejorar confiabilidad de respuestas
- Diseñar prompts few-shot efectivos para guiar comportamiento del agente
- Comparar y contrastar diferentes estrategias de prompting
- **Implementar 5 funciones core de prompting avanzado (100 puntos)**

---

## 📋 Tabla de Contenidos

1. [Motivación: El Poder del Prompting](#1-motivacion)
2. [Intuición Visual: Estrategias de Razonamiento](#2-intuicion-visual)
3. [Fundamentos Matemáticos](#3-fundamentos-matematicos)
4. [Implementación Desde Cero](#4-implementacion-desde-cero)
5. [🎓 Ejercicios Prácticos Guiados (100 pts)](#5-ejercicios-graded)
6. [Comparación de Frameworks](#6-frameworks)
7. [Ejercicios Avanzados (Opcionales)](#7-ejercicios-avanzados)
8. [📄 Papers y Referencias](#8-papers)
9. [💡 Best Practices y Producción](#9-best-practices)
10. [📍 Navegación y Próximos Pasos](#10-navegacion)

---


## 1. Motivación: El Poder del Prompting

### El Problema: LLMs Pueden "Pensar" Mal

Pregunta simple: *"Roger tiene 5 pelotas de tenis. Compra 2 latas más de pelotas. Cada lata tiene 3 pelotas. ¿Cuántas pelotas tiene ahora?"*

**LLM sin prompting especial:**
```
Respuesta: Roger tiene 10 pelotas.
```
❌ **Incorrecto** (5 + 2*3 = 11)

**LLM con Chain-of-Thought:**
```
Pensemos paso a paso:
1. Roger empieza con 5 pelotas
2. Compra 2 latas, cada lata tiene 3 pelotas
3. Pelotas nuevas: 2 × 3 = 6 pelotas
4. Total: 5 + 6 = 11 pelotas

Respuesta: Roger tiene 11 pelotas.
```
✅ **Correcto**

### La Solución: Prompting Agéntico

Técnicas que hacen que LLMs "piensen" mejor:

1. **Chain-of-Thought (CoT)**: Razonamiento secuencial explícito
2. **Tree-of-Thought (ToT)**: Exploración ramificada de posibilidades
3. **Self-Consistency**: Múltiples intentos + votación
4. **Few-Shot Learning**: Aprender de ejemplos concretos

### Pregunta Guía

**Al final de este notebook responderemos:**
*¿Cómo podemos hacer que un LLM razone de forma más confiable y sistemática usando solo el diseño del prompt?*

## 2. Intuición Visual: Estrategias de Razonamiento

### Comparación Visual

```
┌──────────────────────────────────────────────────────────┐
│              PROMPTING ESTÁNDAR (Naive)                 │
├──────────────────────────────────────────────────────────┤
│                                                          │
│  Pregunta ──────► [LLM] ──────► Respuesta              │
│                                                          │
│  • Una sola pasada                                      │
│  • Sin razonamiento explícito                           │
│  • Propenso a errores                                   │
└──────────────────────────────────────────────────────────┘

┌──────────────────────────────────────────────────────────┐
│              CHAIN-OF-THOUGHT (CoT)                     │
├──────────────────────────────────────────────────────────┤
│                                                          │
│  Pregunta ──► [LLM] ──► Paso 1 ──► Paso 2 ──► ... ──►  │
│                           ↓          ↓                   │
│                        Explícito  Explícito              │
│                                      ↓                   │
│                                  Respuesta               │
│                                                          │
│  • Razonamiento paso a paso                             │
│  • Trazabilidad completa                                │
│  • Más confiable                                        │
└──────────────────────────────────────────────────────────┘

┌──────────────────────────────────────────────────────────┐
│              TREE-OF-THOUGHT (ToT)                      │
├──────────────────────────────────────────────────────────┤
│                                                          │
│                      Pregunta                            │
│                         │                                │
│              ┌──────────┼──────────┐                    │
│              ▼          ▼          ▼                    │
│           Camino A   Camino B   Camino C                │
│              │          │          │                    │
│         ┌────┴────┐    │     ┌────┴────┐               │
│         ▼         ▼    ▼     ▼         ▼               │
│       Sub-A1   Sub-A2  ...  Sub-C1   Sub-C2            │
│                                                          │
│  [Evaluación y selección del mejor camino]             │
│                         │                                │
│                         ▼                                │
│                    Respuesta                             │
│                                                          │
│  • Exploración de alternativas                          │
│  • Evaluación de caminos                                │
│  • Backtracking si es necesario                         │
└──────────────────────────────────────────────────────────┘

┌──────────────────────────────────────────────────────────┐
│              SELF-CONSISTENCY                           │
├──────────────────────────────────────────────────────────┤
│                                                          │
│                     Pregunta                             │
│                        │                                 │
│       ┌────────────────┼────────────────┐               │
│       │                │                │               │
│       ▼                ▼                ▼               │
│   [LLM CoT]       [LLM CoT]       [LLM CoT]            │
│   (temp=0.7)      (temp=0.7)      (temp=0.7)           │
│       │                │                │               │
│       ▼                ▼                ▼               │
│  Respuesta A    Respuesta B    Respuesta C              │
│                        │                                 │
│                   [VOTACIÓN]                             │
│                        │                                 │
│                        ▼                                 │
│                Respuesta Final                           │
│                (mayoría gana)                            │
│                                                          │
│  • Múltiples ejecuciones                                │
│  • Mayor confiabilidad                                  │
│  • Costoso (N × llamadas)                               │
└──────────────────────────────────────────────────────────┘
```

In [ ]:
# Instalación de dependencias
# !pip install openai anthropic python-dotenv plotly numpy

import os
import json
import re
from typing import List, Dict, Tuple, Optional
from collections import Counter
from dataclasses import dataclass
import numpy as np

# Visualizaciones
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# APIs
try:
    from openai import OpenAI
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False
    print("⚠️  OpenAI no disponible")

from dotenv import load_dotenv
load_dotenv()

print("✅ Librerías importadas")

## 3. Fundamentos Matemáticos: Teoría del Prompting

### Modelando el Prompting

Sea $\mathcal{L}$ un LLM, formalmente podemos expresar:

$$
\begin{align}
y &= \mathcal{L}(x) \tag{1} \\
\text{donde: } & \\
x &: \text{prompt (input)} \\
y &: \text{respuesta (output)} \\
\mathcal{L} &: \text{función LLM} 
\end{align}
$$

### Chain-of-Thought (CoT)

En lugar de $x \to y$ directamente, descomponemos:

$$
\begin{align}
x &\to r_1 \to r_2 \to ... \to r_n \to y \tag{2} \\
\text{donde: } & \\
r_i &: \text{paso intermedio de razonamiento} \\
n &: \text{número de pasos}
\end{align}
$$

El prompt CoT típicamente incluye: $x_{\text{CoT}} = x + \text{"Let's think step by step"}$

### Self-Consistency

Generamos $k$ respuestas con temperatura $T > 0$:

$$
\begin{align}
\{y_1, y_2, ..., y_k\} &= \{\mathcal{L}(x, T)_i\}_{i=1}^k \tag{3} \\
y_{\text{final}} &= \text{mode}(\{y_1, y_2, ..., y_k\}) \tag{4} \\
\text{donde: } & \\
\text{mode}() &: \text{valor más frecuente (votación)} 
\end{align}
$$

**Probabilidad de correctitud:**

Si cada intento tiene probabilidad $p$ de ser correcto:

$$
P(\text{correcto}) = \sum_{i=\lceil k/2 \rceil}^{k} \binom{k}{i} p^i (1-p)^{k-i} \tag{5}
$$

**Ejemplo numérico:** Si $p=0.7$ (70% accuracy individual) y $k=5$:

$$
P(\text{mayoría correcta}) \approx 0.837 \text{ (83.7%)}
$$

### Tree-of-Thought (ToT)

Modelamos como búsqueda en árbol:

$$
\begin{align}
S(s, a) &= \text{evaluación del estado } s \text{ tras acción } a \tag{6} \\
\text{best\_path} &= \arg\max_{\text{path}} \sum_{s \in \text{path}} S(s) \tag{7}
\end{align}
$$

Donde $S(s)$ puede ser evaluado por el mismo LLM o heurísticas.

**Key Insight:**
> El prompting no cambia las capacidades del modelo, pero sí cómo las usa. Es como la diferencia entre "pensar en voz alta" vs "responder impulsivamente".

## 4. Implementación Desde Cero

### 4.1 Chain-of-Thought (CoT)

In [ ]:
class ChainOfThoughtPrompting:
    """
    Implementación de Chain-of-Thought prompting.
    """
    
    def __init__(self, llm_backend="simulated", model="gpt-4-turbo-preview"):
        self.llm_backend = llm_backend
        self.model = model
        
        if llm_backend == "openai" and OPENAI_AVAILABLE:
            self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        else:
            self.client = None
    
    def _call_llm(self, prompt: str, temperature: float = 0.0) -> str:
        """Llama al LLM"""
        if self.llm_backend == "openai" and self.client:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature
            )
            return response.choices[0].message.content
        else:
            # Simulado
            return self._simulated_cot_response(prompt)
    
    def _simulated_cot_response(self, prompt: str) -> str:
        """Simula una respuesta CoT (solo para demo)"""
        if "pelotas" in prompt.lower():
            return """Pensemos paso a paso:
1. Roger empieza con 5 pelotas de tenis
2. Compra 2 latas más de pelotas
3. Cada lata contiene 3 pelotas
4. Pelotas en las latas: 2 × 3 = 6 pelotas
5. Total: 5 (inicial) + 6 (nuevas) = 11 pelotas

Por lo tanto, Roger tiene 11 pelotas de tenis ahora."""
        return "Respuesta simulada"
    
    def zero_shot_cot(self, question: str) -> str:
        """
        Zero-shot CoT: Solo agrega "Let's think step by step"
        
        Paper: "Large Language Models are Zero-Shot Reasoners" (Kojima et al., 2022)
        """
        prompt = f"{question}\n\nLet's think step by step:"
        return self._call_llm(prompt)
    
    def few_shot_cot(self, question: str, examples: List[Dict[str, str]]) -> str:
        """
        Few-shot CoT: Proporciona ejemplos con razonamiento
        
        Args:
            question: Pregunta a responder
            examples: Lista de {"question": ..., "reasoning": ..., "answer": ...}
        """
        # Construir prompt con ejemplos
        prompt_parts = []
        
        for i, ex in enumerate(examples, 1):
            prompt_parts.append(f"Ejemplo {i}:")
            prompt_parts.append(f"Pregunta: {ex['question']}")
            prompt_parts.append(f"Razonamiento: {ex['reasoning']}")
            prompt_parts.append(f"Respuesta: {ex['answer']}")
            prompt_parts.append("")
        
        # Agregar pregunta actual
        prompt_parts.append(f"Ahora responde esta pregunta:")
        prompt_parts.append(f"Pregunta: {question}")
        prompt_parts.append(f"Razonamiento:")
        
        prompt = "\n".join(prompt_parts)
        return self._call_llm(prompt)

# Crear instancia
cot = ChainOfThoughtPrompting(llm_backend="simulated")

# Probar Zero-Shot CoT
question = "Roger tiene 5 pelotas de tenis. Compra 2 latas más de pelotas. Cada lata tiene 3 pelotas. ¿Cuántas pelotas tiene ahora?"
response = cot.zero_shot_cot(question)

print("🧮 Zero-Shot Chain-of-Thought")
print("="*60)
print(f"Pregunta: {question}")
print(f"\nRespuesta:\n{response}")

### 4.2 Self-Consistency

In [ ]:
class SelfConsistency:
    """
    Implementa Self-Consistency: múltiples muestreos + votación
    
    Paper: "Self-Consistency Improves Chain of Thought Reasoning in Language Models"
    (Wang et al., 2022)
    """
    
    def __init__(self, cot_prompter: ChainOfThoughtPrompting):
        self.cot = cot_prompter
    
    def _extract_final_answer(self, response: str) -> str:
        """
        Extrae la respuesta final del razonamiento CoT.
        
        Busca patrones como:
        - "Por lo tanto, X"
        - "La respuesta es X"
        - Último número mencionado
        """
        # Buscar patrones comunes
        patterns = [
            r'por lo tanto[,:]?\s*(?:tiene|hay|es|son)?\s*(\d+)',
            r'la respuesta es\s*(\d+)',
            r'total[:\s]*(\d+)',
            r'= (\d+)\s*(?:pelotas|unidades)?\s*$'
        ]
        
        for pattern in patterns:
            match = re.search(pattern, response.lower())
            if match:
                return match.group(1)
        
        # Fallback: último número en la respuesta
        numbers = re.findall(r'\d+', response)
        if numbers:
            return numbers[-1]
        
        return response.strip()
    
    def run(
        self, 
        question: str, 
        num_samples: int = 5,
        temperature: float = 0.7,
        verbose: bool = True
    ) -> Dict:
        """
        Ejecuta Self-Consistency.
        
        Args:
            question: Pregunta a responder
            num_samples: Número de muestreos (típicamente 5-40)
            temperature: Temperatura para sampling (típicamente 0.7)
            verbose: Imprimir pasos
        
        Returns:
            Dict con respuesta final, todas las respuestas, y conteos
        """
        if verbose:
            print(f"\n🔄 Self-Consistency con {num_samples} muestreos")
            print("="*60)
        
        # Generar múltiples razonamientos
        all_responses = []
        all_answers = []
        
        for i in range(num_samples):
            if verbose:
                print(f"\n--- Muestreo {i+1}/{num_samples} ---")
            
            # Generar respuesta con CoT
            response = self.cot.zero_shot_cot(question)
            all_responses.append(response)
            
            # Extraer respuesta final
            answer = self._extract_final_answer(response)
            all_answers.append(answer)
            
            if verbose:
                print(f"Respuesta extraída: {answer}")
        
        # Votar por mayoría
        answer_counts = Counter(all_answers)
        most_common_answer, count = answer_counts.most_common(1)[0]
        
        if verbose:
            print(f"\n📊 Votación:")
            for answer, cnt in answer_counts.most_common():
                print(f"  {answer}: {cnt}/{num_samples} votos ({cnt/num_samples*100:.1f}%)")
            print(f"\n✅ Respuesta final (mayoría): {most_common_answer}")
        
        return {
            "final_answer": most_common_answer,
            "all_responses": all_responses,
            "all_answers": all_answers,
            "vote_counts": dict(answer_counts),
            "confidence": count / num_samples
        }

# Probar Self-Consistency
sc = SelfConsistency(cot)
result = sc.run(question, num_samples=5, verbose=True)

print(f"\n🎯 Confianza: {result['confidence']*100:.1f}%")

### 4.3 Tree-of-Thought (ToT)

In [ ]:
class TreeOfThought:
    """
    Implementación simplificada de Tree-of-Thought.
    
    Paper: "Tree of Thoughts: Deliberate Problem Solving with LLMs" (Yao et al., 2023)
    
    ToT permite explorar múltiples caminos de razonamiento y hacer backtracking.
    """
    
    def __init__(self, cot_prompter: ChainOfThoughtPrompting):
        self.cot = cot_prompter
    
    def generate_thoughts(self, problem: str, num_thoughts: int = 3) -> List[str]:
        """
        Genera múltiples "pensamientos" iniciales (caminos posibles).
        """
        prompt = f"""Dado el siguiente problema, genera {num_thoughts} enfoques diferentes para resolverlo.

Problema: {problem}

Lista {num_thoughts} enfoques distintos (solo el primer paso de cada uno):"""
        
        response = self.cot._call_llm(prompt)
        
        # Parsear respuestas (simplificado)
        thoughts = []
        for line in response.split('\n'):
            if line.strip() and (line[0].isdigit() or line.startswith('-')):
                thought = re.sub(r'^[\d.-]+\s*', '', line.strip())
                if thought:
                    thoughts.append(thought)
        
        # Si no se parseó bien, retornar respuesta completa
        if not thoughts:
            thoughts = [f"Enfoque {i+1}": {problem}" for i in range(num_thoughts)]
        
        return thoughts[:num_thoughts]
    
    def evaluate_thought(self, problem: str, thought: str) -> float:
        """
        Evalúa qué tan prometedor es un pensamiento (0-1).
        
        En la práctica, esto podría ser:
        - Evaluación por el LLM
        - Heurística de dominio
        - Modelo separado
        """
        prompt = f"""Evalúa qué tan prometedor es el siguiente enfoque para resolver el problema.

Problema: {problem}

Enfoque: {thought}

Da una calificación de 1-10 donde:
- 1: Muy poco prometedor
- 10: Muy prometedor

Calificación (solo el número):"""
        
        response = self.cot._call_llm(prompt)
        
        # Extraer número
        match = re.search(r'(\d+)', response)
        if match:
            score = int(match.group(1))
            return min(score / 10.0, 1.0)
        
        return 0.5  # Default
    
    def run(
        self,
        problem: str,
        num_thoughts: int = 3,
        depth: int = 2,
        verbose: bool = True
    ) -> Dict:
        """
        Ejecuta ToT con búsqueda en árbol.
        
        Args:
            problem: Problema a resolver
            num_thoughts: Número de pensamientos a generar por nivel
            depth: Profundidad del árbol
            verbose: Imprimir pasos
        """
        if verbose:
            print(f"\n🌳 Tree-of-Thought (profundidad={depth}, breadth={num_thoughts})")
            print("="*60)
        
        # Generar pensamientos iniciales
        thoughts = self.generate_thoughts(problem, num_thoughts)
        
        if verbose:
            print(f"\n📝 Pensamientos generados:")
            for i, t in enumerate(thoughts, 1):
                print(f"  {i}. {t}")
        
        # Evaluar pensamientos
        evaluations = []
        for thought in thoughts:
            score = self.evaluate_thought(problem, thought)
            evaluations.append(score)
        
        if verbose:
            print(f"\n📊 Evaluaciones:")
            for i, (thought, score) in enumerate(zip(thoughts, evaluations), 1):
                print(f"  {i}. Score: {score:.2f} - {thought[:50]}...")
        
        # Seleccionar mejor camino
        best_idx = np.argmax(evaluations)
        best_thought = thoughts[best_idx]
        
        if verbose:
            print(f"\n✅ Mejor camino seleccionado: {best_thought}")
        
        # Continuar con el mejor camino usando CoT
        final_prompt = f"{problem}\n\nEnfoque seleccionado: {best_thought}\n\nResuelve paso a paso:"
        final_response = self.cot._call_llm(final_prompt)
        
        return {
            "all_thoughts": thoughts,
            "evaluations": evaluations,
            "best_thought": best_thought,
            "final_response": final_response
        }

# Probar ToT
tot = TreeOfThought(cot)
result = tot.run(question, num_thoughts=3, verbose=True)

print(f"\n🎯 Respuesta final:")
print(result['final_response'])

## 5. Versión con Framework: LangChain

LangChain incluye soporte para varias de estas técnicas.

In [ ]:
# Ejemplo conceptual con LangChain
# !pip install langchain langchain-openai

try:
    from langchain.prompts import PromptTemplate
    from langchain_openai import ChatOpenAI
    
    # Template para CoT
    cot_template = PromptTemplate(
        input_variables=["question"],
        template="""{question}\n\nLet's solve this step by step:\n"""
    )
    
    # LLM
    llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")
    
    # Chain
    # cot_chain = cot_template | llm
    # response = cot_chain.invoke({"question": question})
    
    print("✅ LangChain disponible")
except ImportError:
    print("⚠️  LangChain no disponible")

## 6. Visualización de Resultados

In [ ]:
def compare_prompting_strategies():
    """
    Compara diferentes estrategias de prompting.
    """
    strategies = ['Naive', 'Zero-Shot CoT', 'Few-Shot CoT', 'Self-Consistency', 'Tree-of-Thought']
    
    # Métricas simuladas (en práctica, medir en benchmark)
    accuracy = [0.65, 0.78, 0.82, 0.87, 0.85]
    cost_factor = [1.0, 1.0, 1.2, 5.0, 8.0]  # Relativo a naive
    latency_factor = [1.0, 1.1, 1.3, 5.2, 4.8]
    
    # Crear subplots
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Accuracy', 'Cost (relative)', 'Latency (relative)', 'Accuracy vs Cost'),
        specs=[[{'type': 'bar'}, {'type': 'bar'}],
               [{'type': 'bar'}, {'type': 'scatter'}]]
    )
    
    # Accuracy
    fig.add_trace(
        go.Bar(x=strategies, y=accuracy, marker_color='#3498db', name='Accuracy'),
        row=1, col=1
    )
    
    # Cost
    fig.add_trace(
        go.Bar(x=strategies, y=cost_factor, marker_color='#e74c3c', name='Cost'),
        row=1, col=2
    )
    
    # Latency
    fig.add_trace(
        go.Bar(x=strategies, y=latency_factor, marker_color='#f39c12', name='Latency'),
        row=2, col=1
    )
    
    # Accuracy vs Cost (Pareto frontier)
    fig.add_trace(
        go.Scatter(
            x=cost_factor, 
            y=accuracy,
            mode='markers+text',
            marker=dict(size=12, color='#2ecc71'),
            text=strategies,
            textposition="top center",
            name='Strategies'
        ),
        row=2, col=2
    )
    
    fig.update_xaxes(title_text="Strategy", row=1, col=1)
    fig.update_xaxes(title_text="Strategy", row=1, col=2)
    fig.update_xaxes(title_text="Strategy", row=2, col=1)
    fig.update_xaxes(title_text="Cost Factor", row=2, col=2)
    
    fig.update_yaxes(title_text="Accuracy", row=1, col=1, range=[0, 1])
    fig.update_yaxes(title_text="Cost Factor", row=1, col=2)
    fig.update_yaxes(title_text="Latency Factor", row=2, col=1)
    fig.update_yaxes(title_text="Accuracy", row=2, col=2, range=[0, 1])
    
    fig.update_layout(
        height=800,
        title_text="Comparación de Estrategias de Prompting",
        showlegend=False,
        template='plotly_white'
    )
    
    return fig

fig = compare_prompting_strategies()
fig.show()

<a id="5-ejercicios-graded"></a>
## 5. 🎓 Ejercicios Prácticos Guiados (100 pts)

Implementarás las funciones core de prompting agéntico avanzado. Cada ejercicio construye sobre técnicas estudiadas.

### 📊 Sistema de Calificación

- **Total de puntos**: 100
- **Mínimo para aprobar**: 70
- **Ejercicios**:
  1. `implement_zero_shot_cot` (15 pts) - Implementar Zero-Shot CoT
  2. `parse_cot_steps` (15 pts) - Parsear pasos de razonamiento
  3. `implement_majority_vote` (20 pts) - Algoritmo de votación mayoría
  4. `evaluate_reasoning_path` (25 pts) - Evaluar calidad de razonamiento
  5. `select_best_examples` (25 pts) - Selección inteligente de ejemplos few-shot

---


In [ ]:
# Importar el sistema de autograding
import sys
sys.path.append('/home/user/TUTORIALS-AI-AGENTS/rutas/03-llm-agents')

from tests.test_02_prompting_agentico import PromptingAgenticoGrader

grader = PromptingAgenticoGrader()

print(f"✅ Autograder cargado")
print(f"📊 Total de puntos: {grader.total_points}")
print(f"🎯 Mínimo para aprobar: {grader.passing_grade}")
print(f"\n📝 Ejercicios:")
for name, points in grader.exercise_points.items():
    print(f"  - {name}: {points} pts")


### Ejercicio 1: Implement Zero-Shot CoT (15 pts)

**Objetivo**: Implementar una función que agregue el prompt CoT a una pregunta.

**Contexto**: Zero-shot CoT es la técnica más simple y efectiva de prompting. Solo requiere agregar "Let's think step by step" u frase similar.

**Tu tarea**: 
- Tomar una pregunta (str)
- Retornar la pregunta con prompt CoT agregado
- Debe incluir frases como "paso a paso", "razona", "piensa", etc.


In [ ]:
# GRADED FUNCTION: implement_zero_shot_cot

def implement_zero_shot_cot(question: str) -> str:
    """
    Implementa Zero-Shot Chain-of-Thought prompting.
    
    Args:
        question: Pregunta del usuario
    
    Returns:
        Pregunta con prompt CoT agregado
    
    Ejemplo:
        >>> implement_zero_shot_cot("¿Cuánto es 2+2?")
        "¿Cuánto es 2+2?\n\nPensemos paso a paso:"
    """
    # Agregar prompt CoT estándar
    cot_prompt = f"{question}\n\nPensemos paso a paso:"
    return cot_prompt

In [ ]:
# Correr tests para Ejercicio 1
print('Testing implement_zero_shot_cot...')
print('-' * 50)

# Test 1: Caso básico
try:
    result = implement_zero_shot_cot("¿Cuánto es 15 + 27?")
    assert "¿Cuánto es 15 + 27?" in result, "Should contain original question"
    assert "paso a paso" in result.lower() or "step by step" in result.lower(), "Should contain CoT trigger"
    print('✅ Test 1 passed: Basic CoT prompt')
except AssertionError as e:
    print(f'❌ Test 1 failed: {e}')

# Test 2: Pregunta compleja
try:
    result = implement_zero_shot_cot("Si Roger tiene 5 pelotas y compra 2 latas de 3 pelotas cada una, ¿cuántas tiene en total?")
    assert isinstance(result, str), "Should return string"
    assert len(result) > len("Si Roger tiene 5 pelotas y compra 2 latas de 3 pelotas cada una, ¿cuántas tiene en total?"), \
        "Should be longer than original"
    print('✅ Test 2 passed: Complex question')
except AssertionError as e:
    print(f'❌ Test 2 failed: {e}')

print('\n✅ +15 pts')

### Ejercicio 2: Parse CoT Steps (15 pts)

**Objetivo**: Parsear los pasos de razonamiento de una respuesta CoT.

**Contexto**: Las respuestas CoT contienen pasos numerados o marcados. Necesitamos extraerlos programáticamente para análisis.

**Tu tarea**: Implementar función que extraiga lista de pasos desde una respuesta CoT.


In [ ]:
# GRADED FUNCTION: parse_cot_steps

def parse_cot_steps(cot_response: str) -> List[str]:
    """
    Parsea pasos de razonamiento de una respuesta CoT.
    
    Args:
        cot_response: Respuesta con razonamiento paso a paso
    
    Returns:
        Lista de pasos (strings)
    
    Ejemplo:
        >>> response = '''1. Primer paso
        ... 2. Segundo paso
        ... 3. Tercer paso'''
        >>> parse_cot_steps(response)
        ['Primer paso', 'Segundo paso', 'Tercer paso']
    """
    import re
    
    steps = []
    # Buscar líneas que empiezan con número seguido de punto o paréntesis
    lines = cot_response.split('\n')
    
    for line in lines:
        # Patron: "1.", "1)", "Paso 1:", etc.
        match = re.match(r'^\s*(?:\d+[.):]\s*|Paso\s*\d+:\s*)(.+)', line, re.IGNORECASE)
        if match:
            steps.append(match.group(1).strip())
        # También considerar líneas con guiones
        elif line.strip().startswith('- '):
            steps.append(line.strip()[2:])
    
    return steps

In [ ]:
# Tests parse_cot_steps
print('Testing parse_cot_steps...')
print('-' * 50)

try:
    response1 = """1. Roger empieza con 5 pelotas
2. Compra 2 latas de 3 pelotas cada una
3. Total nuevo: 2 × 3 = 6
4. Total final: 5 + 6 = 11"""
    
    steps = parse_cot_steps(response1)
    assert len(steps) == 4, f"Expected 4 steps, got {len(steps)}"
    assert "Roger empieza con 5 pelotas" in steps[0]
    print(f'✅ Test passed: Parsed {len(steps)} steps')
    print('\n✅ +15 pts')
except AssertionError as e:
    print(f'❌ Test failed: {e}')

### Ejercicio 3: Implement Majority Vote (20 pts)

**Objetivo**: Implementar algoritmo de votación por mayoría para Self-Consistency.

**Tu tarea**: Tomar lista de respuestas y retornar la más común.


In [ ]:
# GRADED FUNCTION: implement_majority_vote

def implement_majority_vote(responses: List[str]) -> str:
    """
    Implementa votación por mayoría.
    
    Args:
        responses: Lista de respuestas posibles
    
    Returns:
        Respuesta más común (en caso de empate, cualquiera)
    """
    from collections import Counter
    
    # Contar frecuencia de cada respuesta
    counts = Counter(responses)
    
    # Retornar la más común
    most_common = counts.most_common(1)[0][0]
    
    return most_common

In [ ]:
# Tests implement_majority_vote
print('Testing implement_majority_vote...')
print('-' * 50)

try:
    responses = ["11", "11", "11", "10", "12"]
    result = implement_majority_vote(responses)
    assert result == "11", f"Expected '11', got '{result}'"
    print('✅ Test passed: Majority vote works')
    print('\n✅ +20 pts')
except AssertionError as e:
    print(f'❌ Test failed: {e}')

### Ejercicio 4: Evaluate Reasoning Path (25 pts)

**Objetivo**: Evaluar la calidad de un camino de razonamiento (para ToT).

**Tu tarea**: Dar un score 0.0-1.0 a un pensamiento basado en qué tan prometedor parece.


In [ ]:
# GRADED FUNCTION: evaluate_reasoning_path

def evaluate_reasoning_path(problem: str, thought: str) -> float:
    """
    Evalúa qué tan prometedor es un camino de razonamiento.
    
    Args:
        problem: Problema a resolver
        thought: Pensamiento/enfoque propuesto
    
    Returns:
        Score 0.0-1.0 (mayor = más prometedor)
    """
    score = 0.5  # Base score
    
    # Heurística 1: Longitud razonable (no muy corto ni muy largo)
    if 20 < len(thought) < 200:
        score += 0.1
    
    # Heurística 2: Contiene palabras clave de razonamiento
    reasoning_keywords = ['paso', 'primero', 'luego', 'después', 'porque', 'entonces', 'por lo tanto']
    if any(kw in thought.lower() for kw in reasoning_keywords):
        score += 0.2
    
    # Heurística 3: Relacionado con el problema (palabras compartidas)
    problem_words = set(problem.lower().split())
    thought_words = set(thought.lower().split())
    overlap = len(problem_words & thought_words) / max(len(problem_words), 1)
    score += overlap * 0.2
    
    return min(score, 1.0)

In [ ]:
# Tests evaluate_reasoning_path
print('Testing evaluate_reasoning_path...')
print('-' * 50)

try:
    problem = "¿Cuánto es 15% de 340?"
    good_thought = "Primero, convertir 15% a decimal, luego multiplicar por 340"
    bad_thought = "No sé"
    
    score_good = evaluate_reasoning_path(problem, good_thought)
    score_bad = evaluate_reasoning_path(problem, bad_thought)
    
    assert score_good > score_bad, f"Good thought should score higher: {score_good} vs {score_bad}"
    assert 0 <= score_good <= 1, "Score should be between 0 and 1"
    print(f'✅ Test passed: Good={score_good:.2f}, Bad={score_bad:.2f}')
    print('\n✅ +25 pts')
except AssertionError as e:
    print(f'❌ Test failed: {e}')

### Ejercicio 5: Select Best Examples (25 pts)

**Objetivo**: Seleccionar los mejores k ejemplos para few-shot prompting.

**Tu tarea**: Dado un pool de ejemplos, seleccionar los k más relevantes para una query.


In [ ]:
# GRADED FUNCTION: select_best_examples

def select_best_examples(
    query: str,
    example_pool: List[Dict[str, str]],
    k: int
) -> List[Dict[str, str]]:
    """
    Selecciona los k ejemplos más relevantes para la query.
    
    Args:
        query: Pregunta actual
        example_pool: Pool de ejemplos disponibles
        k: Número de ejemplos a seleccionar
    
    Returns:
        Lista de k ejemplos más relevantes
    """
    # Calcular score de relevancia para cada ejemplo
    scored_examples = []
    query_words = set(query.lower().split())
    
    for example in example_pool:
        # Calcular overlap de palabras
        example_text = (example.get('question', '') + ' ' + example.get('answer', '')).lower()
        example_words = set(example_text.split())
        
        overlap = len(query_words & example_words)
        score = overlap / max(len(query_words), 1)
        
        scored_examples.append((score, example))
    
    # Ordenar por score descendente
    scored_examples.sort(key=lambda x: x[0], reverse=True)
    
    # Retornar top-k
    return [ex for score, ex in scored_examples[:k]]

In [ ]:
# Tests select_best_examples
print('Testing select_best_examples...')
print('-' * 50)

try:
    query = "¿Cuánto es el porcentaje de descuento?"
    pool = [
        {"question": "¿Cuánto es 2+2?", "answer": "4"},
        {"question": "¿Qué porcentaje es 15 de 100?", "answer": "15%"},
        {"question": "Calcular descuento del 20%", "answer": "Multiplicar por 0.8"},
    ]
    
    selected = select_best_examples(query, pool, k=2)
    assert len(selected) <= 2, f"Should return at most k examples, got {len(selected)}"
    # El ejemplo 2 y 3 deberían ser más relevantes (tienen "porcentaje" y "descuento")
    print(f'✅ Test passed: Selected {len(selected)} relevant examples')
    print('\n✅ +25 pts')
    print('\n🎉 NOTEBOOK 02 COMPLETADO - 100/100 pts')
except AssertionError as e:
    print(f'❌ Test failed: {e}')

### 📊 Calificación Final

Ejecuta para obtener tu calificación completa:


In [ ]:
# Descomenta para calificación completa:

# grader.grade_all({
#     'implement_zero_shot_cot': implement_zero_shot_cot,
#     'parse_cot_steps': parse_cot_steps,
#     'implement_majority_vote': implement_majority_vote,
#     'evaluate_reasoning_path': evaluate_reasoning_path,
#     'select_best_examples': select_best_examples
# })


## 7. Ejercicios

### 🟢 Ejercicio 1: Few-Shot Examples

Crea ejemplos few-shot efectivos para un problema de razonamiento lógico.

In [ ]:
def ejercicio_1_few_shot():
    """
    Objetivo: Diseñar ejemplos few-shot para mejorar razonamiento
    
    Tarea: Crear 2-3 ejemplos para problemas de "días de la semana"
    Ejemplo: "Si hoy es lunes y viajas 10 días, ¿qué día será?"
    """
    # TODO: Crea una lista de ejemplos
    examples = [
        {
            "question": "...",
            "reasoning": "...",
            "answer": "..."
        },
        # Agrega más ejemplos
    ]
    
    # TODO: Prueba con few_shot_cot
    # cot = ChainOfThoughtPrompting()
    # result = cot.few_shot_cot("Si hoy es miércoles y viajas 15 días, ¿qué día será?", examples)
    
    pass

# ejercicio_1_few_shot()

### 🟡 Ejercicio 2: Optimizar Self-Consistency

Experimenta con diferentes valores de `num_samples` y analiza el tradeoff accuracy vs cost.

In [ ]:
def ejercicio_2_optimize_sc():
    """
    Objetivo: Encontrar el número óptimo de samples para Self-Consistency
    
    Instrucciones:
    1. Prueba con num_samples = [1, 3, 5, 10, 20]
    2. Mide accuracy (si tienes ground truth) y confianza
    3. Grafica accuracy vs cost (cost = num_samples × cost_per_call)
    4. Identifica el "sweet spot"
    """
    # TODO: Tu código aquí
    pass

# ejercicio_2_optimize_sc()

### 🔴 Ejercicio 3: Implementar ReAct con CoT

Combina CoT con el patrón ReAct (Reasoning + Acting) del notebook anterior.

In [ ]:
def ejercicio_3_react_with_cot():
    """
    Objetivo: Integrar CoT en un agente ReAct
    
    Idea:
    - En cada paso de razonamiento, usa CoT
    - Esto debería hacer que el agente "piense mejor" sobre qué herramienta usar
    
    Desafío:
    Modifica SimpleAgent del notebook 01 para que use CoT en cada decisión.
    """
    # TODO: Tu código aquí
    # Pista: En _build_prompt(), agrega instrucciones CoT
    # Pista 2: Esto será explorado más a fondo en notebook 03
    pass

# Este ejercicio es avanzado - prepara para el siguiente notebook

<a id="8-papers"></a>
## 8. 📄 Papers y Referencias

El prompting agéntico es un campo en rápida evolución (2022-2024). Aquí los papers esenciales organizados por técnica.

### 📖 Guía de Lectura

**Para Principiantes**:
1. Chain-of-Thought (Wei et al., 2022) - EL paper fundacional
2. Zero-Shot CoT (Kojima et al., 2022) - Sorprendentemente simple
3. Lilian Weng's blog - Excelente overview

**Para Implementadores**:
1. Self-Consistency (Wang et al., 2022)
2. Tree of Thoughts (Yao et al., 2023)
3. ReAct (Yao et al., 2023)

**Para Investigadores**:
1. Surveys completos
2. Papers de optimización
3. Benchmarks de evaluación

---


### 🏛️ Papers Fundacionales de Chain-of-Thought

1. **Chain-of-Thought Prompting Elicits Reasoning in Large Language Models**
   - Autores: Wei et al., Google Research, 2022
   - Link: https://arxiv.org/abs/2201.11903
   - Citas: 8,000+ (MEGA influential)
   - **Por qué leerlo**: EL paper que empezó todo. Demuestra que agregar razonamiento explícito mejora dramáticamente performance.
   - **Key insight**: "Let's think step by step" no estaba en el training set, pero funciona asombrosamente bien.
   - **Resultados**: +50% accuracy en GSM8K (math), +40% en SVAMP

2. **Large Language Models are Zero-Shot Reasoners**
   - Autores: Kojima et al., University of Tokyo, 2022
   - Link: https://arxiv.org/abs/2205.11916
   - Citas: 3,500+
   - **Por qué leerlo**: Descubrió que puedes hacer CoT SIN ejemplos (zero-shot).
   - **Key insight**: Solo agregar "Let's think step by step" activa razonamiento.
   - **Impacto**: Simplificó enormemente la implementación de CoT.


### 🎯 Self-Consistency y Mejoras a CoT

3. **Self-Consistency Improves Chain of Thought Reasoning**
   - Autores: Wang et al., Google Research + MIT, 2022
   - Link: https://arxiv.org/abs/2203.11171
   - Citas: 2,500+
   - **Por qué leerlo**: Técnica simple pero poderosa: muestrear múltiples veces y votar.
   - **Key insight**: La diversidad de caminos + votación > un solo camino.
   - **Resultados**: +17% sobre CoT vanilla en GSM8K
   - **Tradeoff**: k× más costoso (k = num samples)

4. **Automatic Chain of Thought Prompting**
   - Autores: Zhang et al., 2022
   - Link: https://arxiv.org/abs/2210.03493
   - Citas: 1,200+
   - **Por qué leerlo**: Automatiza la creación de ejemplos few-shot.
   - **Key insight**: Clustering + diversidad > ejemplos manuales.

5. **Least-to-Most Prompting**
   - Autores: Zhou et al., Google Research, 2022
   - Link: https://arxiv.org/abs/2205.10625
   - Citas: 1,000+
   - **Por qué leerlo**: Descompone problemas de fácil → difícil.
   - **Key insight**: Resolver subproblemas simples primero → componer.


### 🌳 Tree of Thoughts y Búsqueda

6. **Tree of Thoughts: Deliberate Problem Solving with Large Language Models**
   - Autores: Yao et al., Princeton + Google DeepMind, 2023
   - Link: https://arxiv.org/abs/2305.10601
   - Citas: 1,200+
   - **Por qué leerlo**: Generaliza CoT a búsqueda en árbol con backtracking.
   - **Key insight**: Explorar múltiples caminos + evaluación + poda.
   - **Benchmarks**: Game of 24, Creative Writing, Crosswords
   - **Implementación**: BFS o DFS sobre pensamientos

7. **Graph of Thoughts (GoT)**
   - Autores: Besta et al., ETH Zürich, 2023
   - Link: https://arxiv.org/abs/2308.09687
   - Citas: 400+
   - **Por qué leerlo**: Generaliza ToT a grafos arbitrarios.
   - **Key insight**: Pensamientos pueden fusionarse, ramificarse, repetirse.

8. **Algorithm of Thoughts (AoT)**
   - Autores: Sel et al., 2023
   - Link: https://arxiv.org/abs/2308.10379
   - **Por qué leerlo**: Reduce llamadas a LLM manteniendo exploración.
   - **Key insight**: Guiar búsqueda con heurísticas.


### 🔧 ReAct: Razonamiento + Acción

9. **ReAct: Synergizing Reasoning and Acting in Language Models**
   - Autores: Yao et al., Princeton + Google Research, 2023
   - Link: https://arxiv.org/abs/2210.03629
   - Citas: 3,500+
   - **Por qué leerlo**: Combina CoT con tool use en patrón unificado.
   - **Key insight**: Alternar Thought → Action → Observation.
   - **Benchmarks**: HotpotQA, FEVER, ALFWorld, WebShop
   - **Impacto**: Base de LangChain, AutoGPT, etc.

10. **Reflexion: Language Agents with Verbal Reinforcement Learning**
    - Autores: Shinn et al., Northeastern + MIT, 2023
    - Link: https://arxiv.org/abs/2303.11366
    - Citas: 900+
    - **Por qué leerlo**: Agentes que aprenden de errores mediante auto-reflexión.
    - **Key insight**: Memory de intentos pasados + razonamiento sobre errores.


### 🎨 Prompting Especializado

11. **Plan-and-Solve Prompting**
    - Autores: Wang et al., 2023
    - Link: https://arxiv.org/abs/2305.04091
    - Citas: 600+
    - **Por qué leerlo**: Mejora CoT separando planning de execution.
    - **Key insight**: "Devise a plan" → "Carry out the plan"

12. **Decomposed Prompting (DeComp)**
    - Autores: Khot et al., AI2, 2022
    - Link: https://arxiv.org/abs/2210.02406
    - **Por qué leerlo**: Descompone tareas complejas en subtareas.
    - **Key insight**: Divide-and-conquer con prompts especializados.

13. **Maieutic Prompting**
    - Autores: Jung et al., 2022
    - Link: https://arxiv.org/abs/2205.11822
    - **Por qué leerlo**: Razonamiento abductivo (inferir explicaciones).
    - **Key insight**: Generar árbol de explicaciones y verificar consistencia.


### 🧪 Evaluación y Benchmarks

14. **Challenging BIG-Bench Tasks and Whether Chain-of-Thought Can Solve Them**
    - Autores: Suzgun et al., Stanford, 2022
    - Link: https://arxiv.org/abs/2210.09261
    - Citas: 800+
    - **Por qué leerlo**: Evalúa CoT en 23 tareas desafiantes.
    - **Key insight**: CoT ayuda más en tareas de razonamiento multi-step.

15. **GSM8K: Training Verifiers to Solve Math Word Problems**
    - Autores: Cobbe et al., OpenAI, 2021
    - Link: https://arxiv.org/abs/2110.14168
    - Citas: 2,000+
    - **Por qué leerlo**: Benchmark estándar para evaluar razonamiento matemático.
    - **Dataset**: 8,500 problemas de matemáticas de escuela primaria.


### 📚 Surveys y Análisis

16. **A Survey on In-context Learning**
    - Autores: Dong et al., Tsinghua, 2022
    - Link: https://arxiv.org/abs/2301.00234
    - Citas: 1,500+
    - **Por qué leerlo**: Survey comprehensivo de few-shot learning.
    - **Scope**: Teoría, métodos, aplicaciones de ICL.

17. **Towards Reasoning in Large Language Models: A Survey**
    - Autores: Huang & Chang, 2022
    - Link: https://arxiv.org/abs/2212.10403
    - Citas: 800+
    - **Por qué leerlo**: Overview de todas las técnicas de razonamiento.
    - **Scope**: CoT, self-consistency, ToT, y más.


### 💡 Optimización y Mejores Prácticas

18. **Large Language Models Can Self-Improve**
    - Autores: Huang et al., Google Research, 2022
    - Link: https://arxiv.org/abs/2210.11610
    - Citas: 600+
    - **Por qué leerlo**: LLMs pueden mejorar sus propias respuestas.
    - **Key insight**: Generar respuestas → filtrar correctas → fine-tune.

19. **STaR: Self-Taught Reasoner**
    - Autores: Zelikman et al., Stanford, 2022
    - Link: https://arxiv.org/abs/2203.14465
    - Citas: 700+
    - **Por qué leerlo**: Bootstrap razonamiento usando respuestas correctas.
    - **Key insight**: Iterar: generar → verificar → entrenar.

20. **Teaching Algorithmic Reasoning via In-context Learning**
    - Autores: Zhou et al., DeepMind, 2022
    - Link: https://arxiv.org/abs/2211.09066
    - **Por qué leerlo**: Cómo enseñar algoritmos mediante ejemplos.


### 🔬 Análisis Teórico

21. **Why Can GPT Learn In-Context?**
    - Autores: Dai et al., 2022
    - Link: https://arxiv.org/abs/2212.10559
    - **Por qué leerlo**: Análisis teórico de por qué funciona ICL.
    - **Key insight**: Attention puede implementar gradient descent.

22. **What Makes Good In-Context Examples for GPT-3?**
    - Autores: Liu et al., UC Irvine, 2021
    - Link: https://arxiv.org/abs/2101.06804
    - Citas: 1,200+
    - **Por qué leerlo**: Guía práctica para seleccionar ejemplos.
    - **Key insights**: Similitud semántica > random, diversidad ayuda.


### 🌟 Aplicaciones Especializadas

23. **Program of Thoughts (PoT)**
    - Autores: Chen et al., CMU, 2022
    - Link: https://arxiv.org/abs/2211.12588
    - Citas: 500+
    - **Por qué leerlo**: CoT pero generando código en vez de texto.
    - **Key insight**: Para math/reasoning, code > texto libre.

24. **Faithful Chain-of-Thought Reasoning**
    - Autores: Lyu et al., 2023
    - Link: https://arxiv.org/abs/2301.13379
    - **Por qué leerlo**: Asegura que razonamiento es faithful (no alucinaciones).
    - **Key insight**: Verificar cada paso con LLM crítico.

25. **Verify-and-Edit: Improving Chain-of-Thought**
    - Autores: Dhuliawala et al., Google Research, 2023
    - Link: https://arxiv.org/abs/2305.03268
    - **Por qué leerlo**: Iterative refinement de razonamiento.
    - **Key insight**: Generate → Verify → Edit → Repeat.


### 📝 Blogs y Recursos Prácticos

26. **Prompt Engineering Guide**
    - Link: https://www.promptingguide.ai/
    - **Por qué leerlo**: Guía práctica completa y actualizada.
    - **Scope**: Todas las técnicas con ejemplos ejecutables.

27. **Lilian Weng - Prompting**
    - Link: https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/
    - **Por qué leerlo**: Overview técnico excelente por investigadora de OpenAI.

28. **OpenAI Cookbook - Techniques**
    - Link: https://cookbook.openai.com/
    - **Por qué leerlo**: Ejemplos prácticos de OpenAI.


<a id="9-best-practices"></a>
## 9. 💡 Best Practices y Producción

### 🎯 Cuándo Usar Cada Técnica

| Técnica | Cuándo Usar | Cuándo NO Usar | Cost | Latency |
|---------|-------------|----------------|------|----------|
| **Naive** | Tareas simples, prototipo | Razonamiento complejo | 1x | 1x |
| **Zero-Shot CoT** | Mayoría de casos, balance ideal | Tareas triviales | 1x | 1.1x |
| **Few-Shot CoT** | Dominios específicos | Sin buenos ejemplos | 1.5x | 1.3x |
| **Self-Consistency** | Alta criticidad, presupuesto OK | Bajo presupuesto | 5-10x | 5x |
| **ToT** | Problemas de búsqueda/planning | Tareas directas | 10-50x | 10x |

### 🏗️ Implementación en Producción

**1. Caching de Prompts**
```python
from functools import lru_cache

@lru_cache(maxsize=1000)
def get_cot_prompt(question: str) -> str:
    # Si la pregunta se repite, usar cache
    return f"{question}\n\nLet's think step by step:"
```

**2. Prompt Versioning**
```python
PROMPTS = {
    "v1": "Let's think step by step",
    "v2": "Let's solve this carefully, step by step",
    "v3": "Let's break this down and reason through it:"
}

def get_prompt(version="v2"):
    return PROMPTS[version]
```

**3. A/B Testing de Prompts**
```python
import random

def ab_test_prompt(user_id: int):
    # 50/50 split
    variant = "A" if user_id % 2 == 0 else "B"
    
    prompts = {
        "A": zero_shot_cot,
        "B": few_shot_cot
    }
    
    return prompts[variant], variant
```


### 🔍 Monitoreo y Debugging

**Métricas a Trackear**:

1. **Task Success Rate** por técnica de prompting
2. **Average # of Steps** en razonamiento CoT
3. **Self-Consistency Agreement** (qué % votan por mayoría)
4. **Cost per Query** (tokens × price)
5. **Latency p50, p95, p99**

**Logging Estructurado**:
```python
import logging
import json

def log_cot_execution(question, response, steps, cost):
    log_data = {
        "technique": "zero_shot_cot",
        "question": question,
        "num_steps": len(steps),
        "response": response,
        "cost_usd": cost,
        "timestamp": datetime.now().isoformat()
    }
    
    logging.info(json.dumps(log_data))
```


### 💰 Optimización de Costos

**1. Cascade de Modelos**
```python
def smart_prompting(question, difficulty="auto"):
    if difficulty == "auto":
        # Clasificar dificultad con modelo barato
        difficulty = classify_difficulty(question)  # GPT-3.5
    
    if difficulty == "easy":
        # Naive con modelo barato
        return cheap_llm(question)
    elif difficulty == "medium":
        # Zero-Shot CoT con modelo medio
        return medium_llm(zero_shot_cot(question))
    else:
        # Self-Consistency con modelo caro
        return expensive_llm_with_sc(question)
```

**2. Early Stopping en Self-Consistency**
```python
def adaptive_self_consistency(question, max_samples=10, confidence_threshold=0.8):
    responses = []
    
    for i in range(max_samples):
        response = cot.zero_shot_cot(question)
        responses.append(extract_answer(response))
        
        # Check if we have strong majority
        if i >= 3:  # Minimum 3 samples
            counts = Counter(responses)
            max_count = counts.most_common(1)[0][1]
            confidence = max_count / len(responses)
            
            if confidence >= confidence_threshold:
                print(f"Early stopping at {i+1} samples (confidence: {confidence:.2f})")
                break
    
    return majority_vote(responses)
```

**3. Batch Processing**
```python
# En vez de 100 llamadas individuales
results = [llm.call(q) for q in questions]  # ❌ Lento

# Batch processing
batch_prompt = "\n\n".join(f"Q{i}: {q}" for i, q in enumerate(questions))
batch_result = llm.call(batch_prompt)  # ✅ Más rápido
```


### ⚠️ Common Pitfalls

**Pitfall 1: Over-prompting**
```python
# ❌ MAL: Demasiado verboso
prompt = f"""{question}

Please think very carefully about this question.
Take your time and reason through it step by step.
Make sure to check your work at each step.
Don't rush to a conclusion.
..."""

# ✅ BIEN: Conciso y efectivo
prompt = f"{question}\n\nLet's think step by step:"
```

**Pitfall 2: Ignorar Parsing Errors**
```python
# ❌ MAL: Assume perfecto
answer = extract_final_answer(cot_response)

# ✅ BIEN: Validar y manejar errores
try:
    answer = extract_final_answer(cot_response)
    validate_answer_format(answer)
except ParseError:
    # Re-prompt o fallback
    answer = fallback_extraction(cot_response)
```

**Pitfall 3: No Validar Self-Consistency**
```python
# ❌ MAL: Asumir que mayoría = correcto
final = majority_vote(responses)

# ✅ BIEN: Verificar confidence
counts = Counter(responses)
max_count = counts.most_common(1)[0][1]
confidence = max_count / len(responses)

if confidence < 0.5:
    logger.warning(f"Low confidence: {confidence}")
    # Tal vez re-ejecutar o escalar a humano
```


<a id="10-navegacion"></a>
## 10. 📍 Navegación y Próximos Pasos

### 🎉 ¡Felicitaciones!

Dominas las técnicas de prompting agéntico:

✅ Chain-of-Thought (Zero-shot y Few-shot)  
✅ Self-Consistency  
✅ Tree of Thoughts  
✅ Selección de ejemplos  
✅ Evaluación de razonamiento  
✅ Best practices de producción  

### 📚 Ruta Sugerida

```
01. Intro a LLM Agents ✅
           ↓
02. Prompting Agéntico ✅ ← Tú estás aquí
           ↓
03. Memory y RAG para Agentes 🎯 ← SIGUIENTE
           ↓
04. Planning y Reasoning Avanzado
           ↓
05. Multi-Agent Systems
           ↓
06. Proyecto Final
```

**[➡️ Siguiente: 03. Memory y RAG para Agentes](03-memory-rag-agents.ipynb)**

En el siguiente notebook:
- Short-term vs Long-term memory
- Vector stores y embeddings
- RAG (Retrieval-Augmented Generation)
- MemGPT architecture
- Conversational memory

---

<div align="center">

## 🎓 Has Completado Prompting Agéntico!

### Respuesta a la Pregunta Guía

*¿Cómo hacer que un LLM razone más confiablemente?*

**Respuesta**: Diseñar prompts que **expliciten el proceso de razonamiento**:
- **CoT**: Descompone en pasos intermedios
- **Self-Consistency**: Promedia múltiples caminos
- **ToT**: Explora alternativas sistemáticamente

El prompting no mejora la inteligencia del modelo,  
pero sí **cómo la usa**.

*Built with ❤️ for AI learners*

</div>
